# Naive Bayes: From First Principles

## 1. Combining Clues to Predict a Class

Imagine an email containing “win,” “prize,” and “claim.” Those words provide evidence for spam, but no single word settles the decision. A legitimate message can discuss a prize, and many spam messages use none of these words. We need a way to combine clues while accounting for how common each class is.

Naive Bayes does this with a probability model. It learns how features behave within each class, then uses those patterns to score a new observation. Unlike a hand-written rule such as “every message containing win is spam,” it combines evidence from the represented features and the class frequencies.

Its simplifying assumption is that the features behave independently **once the class is known**. That assumption is often imperfect, but it reduces a difficult joint-distribution problem to many small estimation problems.

<div style="color:#17324d; background-color:#eaf4ff; border-left:5px solid #2878b5; padding:10px 14px; margin:12px 0;"><strong>Core intuition</strong><br>Start with how common each class is, measure how compatible the observed features are with each class, and choose the class with the strongest combined evidence.</div>

We will use text to introduce the reasoning, then continuous measurements to connect it to the Gaussian Naive Bayes code companion. These are different feature models built around the same Bayes decision rule.

## 2. What the Model Receives and Produces

Naive Bayes is a family of **supervised classification** methods. Training data contain feature rows paired with known class labels. At prediction time, a fitted model receives a feature vector and returns a class label and, in many implementations, a probability estimate for each class.

Let $n$ be the number of training rows, $p$ the number of features, and $K$ the number of classes. Write one observation as $\mathbf x=(x_1,\ldots,x_p)$ and a candidate class as $c$.

| Term | Meaning | Email example |
|---|---|---|
| Class | Category to predict | Spam or legitimate |
| Feature vector | Numerical representation of an observation | Counts of vocabulary words |
| Prior | Class probability before this row's evidence | Overall spam frequency |
| Likelihood | How compatible the observed features are with a class | Frequency pattern of these words in spam |
| Posterior | Class probability after incorporating the evidence | Model probability of spam for this message |

Binary and multiclass tasks use the same principle: calculate a score for every candidate class. The representation matters. Word counts, binary indicators, categories, and continuous measurements call for different likelihood models; choosing a Naive Bayes variant is not merely choosing a name.

## 3. Probability and Conditional Probability

A probability lies between 0 and 1. If 30 of 100 training emails are spam, their empirical class frequency is $30/100=0.30$. We can use this as an estimate of $P(S)$, where $S$ is the event “the email is spam.”

A conditional probability asks what happens within a specified subset. For events $A$ and $B$ with $P(B)>0$,

$$
P(A\mid B)=\frac{P(A\cap B)}{P(B)}.
$$

The vertical bar means “given,” and $A\cap B$ means both events occur. For example, $P(W\mid S)$ is the proportion of spam emails containing a particular word, while $P(S\mid W)$ is the proportion of word-containing emails that are spam. These use different denominators and are generally different.

Suppose 12 of the 30 spam emails contain “win,” while 7 of the 70 legitimate emails contain it. Then $P(W\mid S)=12/30=0.4$, but $P(S\mid W)=12/19\approx0.632$. A word can be strongly associated with spam without making spam certain.

This example treats word presence as an event at the document level. Later, the multinomial model will count word tokens instead. Keeping these event definitions distinct prevents incorrect denominators.

## 4. Bayes' Theorem Reverses the Conditioning

Bayes' theorem follows by expressing the same joint probability in two ways:

$$
P(A\cap B)=P(B\mid A)P(A)=P(A\mid B)P(B).
$$

Rearranging gives

$$
P(A\mid B)=\frac{P(B\mid A)P(A)}{P(B)}.
$$

For a class and feature vector, this becomes

$$
P(c\mid\mathbf x)=\frac{P(\mathbf x\mid c)P(c)}{P(\mathbf x)}.
$$

The numerator combines likelihood and prior. The denominator, called the evidence, sums the class-weighted likelihoods across all possible classes and ensures the posterior probabilities add to one.

In the word-presence example,

$$
P(W)=0.4(0.3)+0.1(0.7)=0.19,
$$

so $P(S\mid W)=0.4(0.3)/0.19=12/19$. The legitimate class contributes to the denominator too. Looking only at how frequent a clue is in spam would miss this comparison.

For continuous features, the corresponding expressions use probability densities for the feature observations. The final class posterior remains a probability; we will make the density distinction explicit in the Gaussian section.

## 5. What “Naive” Actually Assumes

Estimating $P(\mathbf x\mid c)$ directly can require enormous amounts of data. Even binary features have $2^p$ possible combinations. Naive Bayes simplifies the joint likelihood by assuming conditional independence:

$$
P(x_1,\ldots,x_p\mid c)=\prod_{j=1}^{p}P(x_j\mid c).
$$

The product symbol means multiply over the features. This assumption concerns dependence **within each class**, not independence across the entire dataset. Two measurements can be associated overall because both differ between classes, even when they are independent within each class.

Real features often remain dependent after the class is known. “New” and “York” co-occur within document categories, and flower length and width can remain related within a species. Treating redundant clues independently may count similar evidence more than once and produce overconfident probabilities.

For multinomial text models, the standard event model instead treats token draws as independent given the class and document length. The resulting count vector follows a multinomial distribution; its component counts are not literally independent at fixed length. This distinction explains why its score uses count-weighted word probabilities.

<div style="color:#664b12; background-color:#fff5df; border-left:5px solid #b7791f; padding:10px 14px; margin:12px 0;"><strong>Common pitfall</strong><br>The method can classify well even when independence is imperfect. That does not mean its probability estimates are calibrated or that duplicated evidence is harmless.</div>

## 6. From Likelihoods to Decisions: Use Log Scores

For independent feature likelihoods, define an unnormalized class score

$$
s_c=P(c)\prod_{j=1}^{p}P(x_j\mid c).
$$

The predicted class is $\hat c=\arg\max_c s_c$: choose the class with the largest score. The evidence denominator is shared across classes, so it does not affect that choice. To obtain posterior probabilities, normalize:

$$
P(c\mid\mathbf x)=\frac{s_c}{\sum_{k=1}^{K}s_k}.
$$

Multiplying many small values can produce numerical underflow, where a positive result becomes zero in computer arithmetic. Implementations therefore work with logarithms:

$$
\ell_c=\log P(c)+\sum_{j=1}^{p}\log P(x_j\mid c).
$$

The logarithm preserves score ordering and turns products into sums. For stable normalization, subtract $M=\max_k\ell_k$ before exponentiation:

$$
P(c\mid\mathbf x)=\frac{\exp(\ell_c-M)}{\sum_k\exp(\ell_k-M)}.
$$

Here $\exp$ is the exponential function. Subtracting the same constant from every log score leaves the probabilities unchanged. Exact zeros in a likelihood still require an appropriate model or smoothing; logarithms alone do not repair them.

## 7. Choose the Likelihood to Match the Features

| Variant | Feature representation | What it models |
|---|---|---|
| Gaussian | Continuous measurements | A separate normal density for each feature within each class |
| Multinomial | Nonnegative counts, often word counts | Class-specific frequencies of vocabulary items |
| Bernoulli | Binary presence/absence indicators | Class-specific chance of each indicator being one |
| Categorical | Discrete categories for each feature | Class-specific probability of each category |
| Complement | Often nonnegative text features | Uses statistics outside each class to form a related text classifier |

The [scikit-learn Naive Bayes guide](https://scikit-learn.org/stable/modules/naive_bayes.html) documents these variants. Complement Naive Bayes is a related approach worth comparing for imbalanced text data, rather than simply substituting another density into every equation above.

Continuous does not automatically mean Gaussian: the within-class distributions may be skewed, multimodal, or heavily affected by outliers. Likewise, arbitrary numeric category codes should not be treated as continuous measurements without justification.

The rest of the lesson develops the three most common variants in enough detail to see their different denominators and assumptions.

## 8. Multinomial Naive Bayes and Smoothing

Represent a document by counts $x_j$ for a fixed vocabulary of size $V$. Let $N_{cj}$ be the total occurrences of vocabulary word $j$ in training documents of class $c$, and $N_c=\sum_jN_{cj}$ the total vocabulary-token count in that class.

Estimate each token probability using additive smoothing:

$$
\theta_{cj}=\frac{N_{cj}+\alpha}{N_c+\alpha V},\qquad\alpha>0.
$$

The probabilities sum to one because the denominator adds the same total pseudocount mass as the numerator. $\alpha=1$ is Laplace smoothing; other positive choices are additive smoothing. Alpha is a hyperparameter to validate, not a universal constant.

Without smoothing, an unseen word within a class receives zero probability and can eliminate that class's score. Smoothing reserves some probability for words not observed in that class. It does not create entries for words outside the fitted vocabulary; handling those is part of the text representation.

For a document of length $L=\sum_jx_j$, the multinomial likelihood is

$$
P(\mathbf x\mid c,L)=\frac{L!}{\prod_jx_j!}\prod_{j=1}^{V}\theta_{cj}^{x_j}.
$$

The factorial term counts token orderings and is the same for every class for a fixed document. Standard class comparison can omit it, giving

$$
\ell_c=\log P(c)+\sum_jx_j\log\theta_{cj}.
$$

Repeated words therefore repeat their contribution. Standard use does not separately model class-specific document lengths. A missing word has $x_j=0$ and contributes zero to this log score, unlike the explicit absence terms in Bernoulli Naive Bayes.

## 9. Worked Text Example: Counts to Posterior

Suppose the training collection has 4 spam and 6 legitimate documents. Our toy vocabulary contains only “win,” “prize,” and “meeting.” Token totals within these classes are:

| Word | Spam count | Legitimate count |
|---|---:|---:|
| win | 4 | 0 |
| prize | 3 | 1 |
| meeting | 1 | 7 |
| Total vocabulary tokens | 8 | 8 |

Class priors use **document counts**, so $P(S)=4/10=0.4$ and $P(H)=6/10=0.6$, where $H$ means legitimate. Word probabilities use **token counts**. With $\alpha=1$ and $V=3$, both token denominators are $8+3=11$:

| Word | $\theta_{Sj}$ | $\theta_{Hj}$ |
|---|---|---|
| win | $5/11$ | $1/11$ |
| prize | $4/11$ | $2/11$ |
| meeting | $2/11$ | $8/11$ |

For the new message “win prize,” the count vector is $(1,1,0)$. Dropping the common multinomial coefficient, calculate

$$
s_S=\frac25\frac5{11}\frac4{11}=\frac8{121},\qquad
s_H=\frac35\frac1{11}\frac2{11}=\frac6{605}.
$$

Using a common denominator, the scores are $40/605$ and $6/605$. Therefore

$$
P(S\mid\mathbf x)=\frac{40}{46}=\frac{20}{23}\approx0.8696,
\qquad P(H\mid\mathbf x)=\frac3{23}\approx0.1304.
$$

The model predicts spam despite the larger legitimate prior. Without smoothing, the unseen “win” token would force the legitimate score to zero. These numbers illustrate the model calculation, not a measured real-world reliability rate.

<div style="color:#17324d; background-color:#eaf4ff; border-left:5px solid #2878b5; padding:10px 14px; margin:12px 0;"><strong>Worked-example takeaway</strong><br>Priors count documents; multinomial likelihoods count tokens. Normalize the combined scores to obtain model probabilities, and keep those probabilities separate from claims about calibration.</div>

## 10. Bernoulli Naive Bayes: Absence Also Counts

Bernoulli features are binary: $x_j=1$ if an indicator is present and $x_j=0$ otherwise. Let $D_c$ be the number of training documents in class $c$ and $D_{cj}$ the number containing word $j$. The smoothed presence estimate is

$$
\pi_{cj}=\frac{D_{cj}+\alpha}{D_c+2\alpha}.
$$

The denominator has $2\alpha$ because there are two outcomes for each feature: present and absent. It does not use the vocabulary-size denominator from the multinomial model.

The class-conditional likelihood is

$$
P(\mathbf x\mid c)=\prod_j\pi_{cj}^{x_j}(1-\pi_{cj})^{1-x_j}.
$$

For a two-feature vector “win present, meeting absent,” its likelihood includes $\pi_{c,\mathrm{win}}(1-\pi_{c,\mathrm{meeting}})$. If those presence probabilities in a class are 0.8 and 0.2, the product is $0.8(0.8)=0.64$. Ignoring the absent feature would give a different model.

Repeating “win” ten times still produces presence value one. Binarizing counts and using Multinomial Naive Bayes is not generally equivalent to Bernoulli Naive Bayes, because the latter includes absence evidence. Compare representations using validation rather than assuming one is always superior.

## 11. Gaussian Naive Bayes: Continuous Measurements

The companion classifies flowers from continuous measurements. Gaussian Naive Bayes assumes that each feature, within each class, follows a normal distribution with its own mean $\mu_{cj}$ and variance $\sigma_{cj}^2$:

$$
f(x_j\mid c)=\frac1{\sqrt{2\pi\sigma_{cj}^2}}
\exp\left(-\frac{(x_j-\mu_{cj})^2}{2\sigma_{cj}^2}\right).
$$

Here $\pi$ is the mathematical circle constant, unrelated to the Bernoulli parameters in the previous section. The exponential decreases as the observation moves away from the class mean relative to its spread. The factor outside the exponential normalizes the density.

For $n_c$ training rows in class $c$, maximum-likelihood estimates are

$$
\mu_{cj}=\frac1{n_c}\sum_{i:y_i=c}x_{ij},\qquad
\sigma_{cj}^2=\frac1{n_c}\sum_{i:y_i=c}(x_{ij}-\mu_{cj})^2.
$$

The variance denominator here is $n_c$, not the $n_c-1$ correction sometimes used for an unbiased sample-variance estimate. The class prior can be estimated as $n_c/n$.

A density value is not a point probability: it can exceed one, while the probability of any one exact value under a continuous distribution is zero. Probabilities over intervals are areas under the density. For classification, multiply the densities by the class prior and normalize across classes to obtain class probabilities.

Gaussian Naive Bayes does not require the entire dataset to look normal. Its assumption applies to each feature separately within each class, together with the conditional-independence assumption. It does not estimate cross-feature covariance within a class.

## 12. Worked Gaussian Example: Fit and Classify

Consider one hypothetical measurement and two classes. Class A has training values 1 and 3; class B has values 3 and 5. Each class has two rows, so both priors are $1/2$.

For class A, the mean is 2 and the maximum-likelihood variance is $[(1-2)^2+(3-2)^2]/2=1$. For class B, the mean is 4 and the variance is also 1.

For a new value $x=2$, the densities are

$$
f(2\mid A)=\frac1{\sqrt{2\pi}}\approx0.39894,
\qquad f(2\mid B)=\frac1{\sqrt{2\pi}}e^{-2}\approx0.05399.
$$

Multiplying by equal priors gives approximately 0.19947 and 0.02700. Normalize:

$$
P(A\mid x=2)=\frac1{1+e^{-2}}\approx0.88080,
\qquad P(B\mid x=2)\approx0.11920.
$$

Thus the model predicts A. At $x=3$, the value is equally far from both means and the priors are equal, so the two model probabilities are equal. A class-returning implementation needs a tie rule; the statistical evidence here does not favor either class.

This hand calculation uses no added variance stabilization and only one feature. Real GaussianNB fitting adds a small variance term to avoid unstable calculations with tiny or zero variances. Two observations per class are also far too few to establish that a Gaussian shape is appropriate; the example demonstrates arithmetic, not model validation.

## 13. Training and Important Design Choices

A basic fitting process is: choose a representation and variant, estimate class priors, estimate each class's feature distribution, then store those estimates. Prediction evaluates each class's log score and normalizes or selects the largest.

| Choice | What it changes | How to assess it |
|---|---|---|
| Variant and feature representation | The assumed likelihood model | Match data meaning, then validate |
| Additive smoothing `alpha` | Strength of count pseudocounts | Tune on training-only validation |
| Gaussian `var_smoothing` | Added variance for numerical stability | Check near-constant features and validate sensible settings |
| Class priors | Starting class probabilities | Use representative frequencies or justified prior knowledge |
| Vocabulary and tokenization | Which text clues are represented | Fit vocabulary on training data and validate preprocessing |
| Decision threshold | Tradeoff between types of classification error | Select for the application's costs using validation |

In scikit-learn GaussianNB, `var_smoothing` scales the largest feature variance in the fitting data to form an additive variance term. It is different from adding token pseudocounts. Supplied priors can replace empirical class frequencies; equal priors are a modeling choice, not an automatic cure for class imbalance.

Gaussian Naive Bayes is not a distance-based model and does not generally need scaling for the reason KNN does. In the ideal unstabilized Gaussian model, consistent invertible per-feature affine rescaling cancels out of class comparisons. A shared numerical variance floor can break that exact invariance, so extreme unit differences still deserve attention.

Multinomial inputs must be nonnegative. Do not apply mean-centering that creates negative text counts. Nonnegative TF–IDF can be useful in practice, but its values are not literal integer token counts; validate it as a representation choice rather than claiming the exact count-generating story still holds.

## 14. A Leakage-Free Practical Workflow

1. Define labels, available prediction-time features, and the primary metric. Decide which class errors are most costly.
2. Reserve a final test set. Stratify when appropriate to preserve class representation, but use group-aware or time-aware splitting when rows are dependent.
3. Within development training data, fit imputation, text vocabulary, transformations, and the classifier in a pipeline. Validation rows must not influence those learned steps.
4. Compare appropriate variants, preprocessing choices, and smoothing settings with cross-validation. Inspect minority-class performance, not only overall accuracy.
5. If tuning thresholds or calibrating probabilities, do so using training-only validation or cross-fitting, without reusing the final test set.
6. Refit the chosen procedure on the development data and evaluate once on the held-out test set.
7. Apply identical preprocessing and class-label mappings to new observations; monitor whether class frequencies or feature patterns change.

A normal random split can leak near-duplicate emails, messages from the same thread, or repeated subject measurements across sets. Good separation depends on the use case, not just a random seed.

<div style="color:#22543d; background-color:#edf8f1; border-left:5px solid #2f855a; padding:10px 14px; margin:12px 0;"><strong>Practical tip</strong><br>Learn the text vocabulary and any feature selection inside each training fold. A fast classifier can still yield an optimistic evaluation if preprocessing or related observations leak across the split.</div>

Missing values need a strategy suited to the estimator and data. A missing measurement is not automatically equivalent to zero, and an unknown category is not automatically a category observed in training.

## 15. Connecting to the Iris Code Companion

`09_Naive_Bayes.ipynb` loads `load_iris(as_frame=True)`, uses the continuous feature columns as inputs and species labels as targets, and makes an 80/20 split with `random_state=42` and `stratify=y`. Stratification preserves class representation; it is not feature preprocessing.

The notebook fits `GaussianNB()` and inspects `class_prior_`, the fitted class probabilities before the test-row measurements are considered. `predict` returns class labels, while `predict_proba` returns class-probability columns in `classes_` order. The companion maps Iris's numeric labels to species names; that convenient ordering should not be assumed for an arbitrary dataset.

It then prints accuracy, a confusion matrix, and a classification report. With the usual scikit-learn convention, confusion-matrix rows represent true classes and columns represent predicted classes. The report gives per-class precision, recall, F1, and support, the number of true observations in each class.

Continuous measurements make GaussianNB a plausible baseline, but do not prove that within-species normality and independence hold. These notes describe the companion's source code without claiming observed accuracy, fitted priors, or probability quality; the companion was not executed for this revision.

The text examples explain other members of the same classifier family. They are not a description of the Iris notebook's preprocessing, which does not build a vocabulary or estimate token probabilities.

## 16. Evaluate Decisions Separately from Probabilities

For a binary task, choose a positive class such as spam. A true positive (TP) is correctly detected spam, a false positive (FP) is legitimate mail flagged as spam, a false negative (FN) is missed spam, and a true negative (TN) is correctly retained legitimate mail.

$$
\mathrm{Accuracy}=\frac{TP+TN}{TP+TN+FP+FN},\quad
\mathrm{Precision}=\frac{TP}{TP+FP},\quad
\mathrm{Recall}=\frac{TP}{TP+FN},
$$

$$
F_1=\frac{2TP}{2TP+FP+FN}.
$$

Precision asks how trustworthy positive predictions are; recall asks how many positives were found. F1 combines the two without including true negatives. Some denominators can be zero, so an evaluation must use and report a defined handling policy.

For example, if TP=8, FP=2, FN=4, and TN=86, accuracy is 94%, precision is 80%, recall is about 66.7%, and F1 is about 72.7%. The high accuracy hides four missed positives. These illustrative counts are unrelated to the Iris companion's outputs.

For multiclass data, inspect each class and distinguish macro averages, which weight classes equally, from support-weighted averages. ROC-AUC assesses ranking and does not require calibrated probabilities; precision–recall analysis can be especially useful when positives are rare. Neither replaces checking a chosen operating threshold.

**Calibration** asks whether, among predictions near 0.8, the event actually occurs about 80% of the time. Conditional-dependence violations can distort Naive Bayes probabilities even when the top-ranked class is useful. Use held-out calibration curves and proper probability scores such as log loss when probability quality matters. Fit any calibration procedure without using final test labels.

<div style="color:#664b12; background-color:#fff5df; border-left:5px solid #b7791f; padding:10px 14px; margin:12px 0;"><strong>Probability is a separate claim</strong><br>A confident model score is not a guarantee. Evaluate ranking, class decisions, and calibration according to how the output will actually be used.</div>

## 17. Strengths, Limitations, Alternatives, and Recap

Naive Bayes can be inexpensive to fit because it estimates small class-specific distributions instead of a full joint distribution. For dense Gaussian data, fitting sufficient statistics takes roughly $O(np)$ work, storing means and variances takes $O(Kp)$ space, and scoring a new row takes $O(Kp)$ work. Text implementations can exploit sparse counts; vocabulary size and class count still affect storage.

| Situation | Why it matters | Useful response or comparison |
|---|---|---|
| Strong dependence within classes | Evidence may be counted redundantly | Inspect correlated features and compare other classifiers |
| Wrong likelihood shape | Class scores misrepresent feature compatibility | Transform carefully or choose another variant/model |
| Rare or unseen count events | Unsmooth estimates can become zero | Validate additive smoothing and vocabulary handling |
| Tiny Gaussian variances | Densities can become numerically unstable | Inspect constant features and variance stabilization |
| Prior or population shift | Training frequencies may not match deployment | Monitor data and reassess priors and calibration |
| Complex feature interactions | Independent likelihoods may miss the decision pattern | Compare trees or other interaction-capable models |

Logistic Regression learns a conditional decision model directly and is a useful comparison for many text and tabular classification tasks. Trees can capture explicit interactions; KNN uses local similarity and has different scaling and prediction-cost concerns. No comparison makes Naive Bayes universally better or worse.

The learned priors, token likelihoods, and Gaussian means describe associations in the training representation. They are not causal effects. Small samples can still give poor estimates, and a fast fit does not guarantee that little data will suffice for the task.

The central sequence is prior, likelihood, combination, and normalization. Choose a likelihood appropriate to the features, use smoothing or variance stabilization where needed, and preserve a clean validation design. Remember the main caution: a useful predicted class and a trustworthy probability estimate are different outcomes that must be evaluated separately.